# Classify new T12 docs

For each Excel file in `SOURCE_DIR`, decide one of:

- **`mapped`** — `.xlsm` (the client's reviewed deal-model workbook containing T12 mapping) → import to `data/mapped_T12_docs/`
- **`raw`**    — `.xlsx` / `.xls` with a T12-style sheet → import to `data/input_T12_docs/`
- **`other`**  — everything else (rent rolls, RedIQ-processed outputs, etc.) → skip

Update `SOURCE_DIR` below and run all cells.

In [1]:
from pathlib import Path
import openpyxl

# ── Tweak this ────────────────────────────────────────────────────────────────
SOURCE_DIR = Path("/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12")

REPO_DATA = Path("/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/app/t12_pipeline/data")
EXISTING_MAPPED = {p.name for p in (REPO_DATA / "mapped_T12_docs").glob("*.xlsm") if not p.name.startswith("~$")}
EXISTING_RAW    = {p.name for p in (REPO_DATA / "input_T12_docs").glob("*.xls*") if not p.name.startswith("~$")}
print(f"Source dir: {SOURCE_DIR}")
print(f"Already in repo:  mapped={len(EXISTING_MAPPED)}  raw={len(EXISTING_RAW)}")

Source dir: /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12
Already in repo:  mapped=19  raw=7


## Classifier — extension first, sheet name second

1. `.xlsm` → **mapped**. Project convention: every `.xlsm` the client sends is a deal-model workbook containing their T12 review, used to build the rulebook.
2. `.xlsx` / `.xls` with a sheet named `T12` / `Trailing` / `Income Statement` → **raw**.
3. Anything else → **other**.

`.xls` files (openpyxl can't open them) fall back to filename matching.

In [2]:
T12_SHEET_KWS = {"t12", "t-12", "trailing", "income statement", "income stat"}


def classify(path: Path) -> str:
    suffix = path.suffix.lower()

    # Rule 1: .xlsm → mapped
    if suffix == ".xlsm":
        return "mapped"

    # Rule 2a: .xls — openpyxl can't read these, fall back to filename
    if suffix == ".xls":
        n = path.name.lower()
        return "raw" if ("t12" in n or "t-12" in n or "trailing" in n) else "other"

    # Rule 2b: .xlsx — open and look at sheet names
    if suffix == ".xlsx":
        try:
            wb = openpyxl.load_workbook(path, read_only=True, data_only=False)
        except Exception:
            return "other"
        is_t12 = any(
            any(kw in s.lower() for kw in T12_SHEET_KWS)
            for s in wb.sheetnames
        )
        wb.close()
        return "raw" if is_t12 else "other"

    return "other"

## Walk + classify

In [3]:
EXCEL = {".xlsx", ".xlsm", ".xls"}

buckets = {"mapped": [], "raw": [], "other": []}
for p in SOURCE_DIR.rglob("*"):
    if not p.is_file() or p.suffix.lower() not in EXCEL:
        continue
    if p.name.startswith("~$") or p.name == ".DS_Store":
        continue
    buckets[classify(p)].append(p)

for kind in ("mapped", "raw", "other"):
    print(f"{kind:7s}: {len(buckets[kind])}")

/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Source Data'!$B$4:INDEX('Source Data'!$P$12:$P$268,COUNTA('Source Data'!$B$12:$B$268)).
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'Source Data'!$B$4:INDEX('Source Data'!$P$12:$P$448,COUNTA('Source Data'!$B$12:$B$448)).
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/manikantamb/Library/Python/3.14/lib/python/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


mapped : 24
raw    : 14
other  : 48


## Mapped files (→ `data/mapped_T12_docs/`)

`*` marks files already in the repo.

In [4]:
new = [p for p in buckets["mapped"] if p.name not in EXISTING_MAPPED]
dup = [p for p in buckets["mapped"] if p.name in EXISTING_MAPPED]
print(f"{len(new)} NEW  +  {len(dup)} already in repo\n")
for p in sorted(buckets["mapped"], key=lambda x: x.name):
    flag = "*" if p.name in EXISTING_MAPPED else " "
    print(f"  {flag} {p.name}")

5 NEW  +  19 already in repo

  * 02242026_Grande Reserve at Geist_V3- VM.xlsm
  * 02252026_Willows of Coventry_V4 (1).xlsm
  * 03092026_The Flats at Arrowood_V4 (FULL BUDGET).xlsm
  * 03092026_The Tatum_V4 (assumption).xlsm
    030926_Tides At Eastchase_v3.xlsm
  * 031026_Mylo Apartment Homes_V2 - VM.xlsm
  * 03112026_Forma at the Park_V1.xlsm
    03112026_The Rivi_V6 - AW Capex Notes .xlsm
  * 03122025_District at Hurstborne_V2.xlsm
  * 03182026_The Park at LeBlanc_V3- VM.xlsm
    03192026_The Jasmine Apartments_V1.xlsm
  * 03192026_The Jasmine Apartments_V2 - VM.xlsm
    03192026_The Park at LeBlanc_V4.1 - AW CapEx.xlsm
  * 03242026_Prescott Park Apartments_V1.5.xlsm
  * 03272026_Devon on Northgate_V1.xlsm
  * 04032026_Villa Del Mar_V1.5.xlsm
  * 080425_Jamestown at St. Matthews_v4.xlsm
  * 080825_Kenton Reserve_v4.xlsm
  * 11182025_Exchange 7272_V1.xlsm
  * 11182025_The Hangar_V1.xlsm
  * 12042025_The Hathaway at Willow Bend_V1.xlsm
  * 12182025_HUB of New Albany_V1.xlsm
  * 260312

## Raw files (→ `data/input_T12_docs/`)

In [5]:
for p in sorted(buckets["raw"], key=lambda x: x.name):
    flag = "*" if p.name in EXISTING_RAW else " "
    print(f"  {flag} {p.name}")

    100325_Park at Eagle Creek_v3.xlsx
    Arbor Gates at Buckhead Seller T12.xlsx
  * Brownsburg Crossing T12 05-25.xlsx
    Creekside Seller T12.xlsx
    Crystal Lake_T-12_04.2026.xlsx
    Ladera Seller T12.xlsx
    Lockhart Seller T-12.xlsx
  * Pebble Brook Village T12 03-25.xlsx
  * Pebble Brook Village T12 09-25.xlsx
    RESERVE AT WHITE ROCK Seller T12.xls
    River Point Seller T12 Feb.xlsx
    River Point West Seller T12.xlsx
    TRAILS AT WHITE ROCK Seller T12.xls
    Wythe Seller T12.xlsx


## Other (skip)

In [6]:
for p in sorted(buckets["other"], key=lambda x: x.name):
    print(f"    {p.name}")

    02242026_Grande Reserve at Geist_V3- VM..xlsx
    Arbor Gates at Buckhead Model T12.xlsx
    Cardinal Model T12.xlsx
    Cardinal Seller T12.xlsx
    Carmel Woods T12 09-25.xlsx
    Chateau in the  Woods T12 08-25.xlsx
    Copy of Prescott Park - RR - 3.20.26.xlsx
    Creekside Apartment Homes Model T12.xlsx
    Devon Rent Roll 3.11.2026.xlsx
    Echo Park at Perry Crossing T12 12-24.xlsx
    Eleven11 Lexington at Flower Mound Model T12.xlsx
    Eleven11 Lexington at Flower Mound Seller T12.xlsx
    English Village RR 12-17-25.xlsx
    Exchange 7272 - RR - 10.15.25.xlsx
    Forma at the Park - RR - 3.2.26.xlsx
    Golfview Manor RR 04-22-2024.xlsx
    Golfview Manor RR 04-22-24 RedIQ.xlsx
    Grande Reserve at Geist Model T12.xlsx
    Hangar RR 04-22-24 RedIQ.xlsx
    Hangar RR 04-22-24.xlsx
    Hathaway Jan26 RR.xlsx
    Holden Model T12.xlsx
    Holden Seller T12.xlsx
    Hub of New Albany RR 08-11-24.xlsx
    Jamestown at St. Matthews - RR - 6.23.25.xlsx
    Jasmine - RR - 3.4.2

## Copy NEW mapped files into the repo

Copies only the `.xlsm` files that aren't already in `data/mapped_T12_docs/`. Already-present files are skipped automatically.

Set `DRY_RUN = False` to actually perform the copy. With `DRY_RUN = True` (default) it just prints what it would do — safe to run.

In [8]:
import shutil

DRY_RUN = False   # flip to False to actually copy

target_dir = REPO_DATA / "mapped_T12_docs"
new_mapped = [p for p in buckets["mapped"] if p.name not in EXISTING_MAPPED]

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Will copy {len(new_mapped)} new .xlsm file(s) -> {target_dir}\n")
for src in sorted(new_mapped, key=lambda x: x.name):
    dst = target_dir / src.name
    print(f"  + {src.name}")
    if not DRY_RUN:
        shutil.copy2(src, dst)

if DRY_RUN:
    print("\n(no files copied — set DRY_RUN = False and re-run to perform the copy)")
else:
    print(f"\nDone. Total in {target_dir.name}: {len(list(target_dir.glob('*.xlsm')))}")

Will copy 5 new .xlsm file(s) -> /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/app/t12_pipeline/data/mapped_T12_docs

  + 030926_Tides At Eastchase_v3.xlsm
  + 03112026_The Rivi_V6 - AW Capex Notes .xlsm
  + 03192026_The Jasmine Apartments_V1.xlsm
  + 03192026_The Park at LeBlanc_V4.1 - AW CapEx.xlsm
  + Golfview Manor - 03.31.2025.xlsm

Done. Total in mapped_T12_docs: 24
